# Global Weather Trend Forecasting

## PM Accelerator Tech Assessment

---

### PM Accelerator Mission

> **"We are committed to offering free Product Management education to teenagers from underserved families. Our mission is to break down financial barriers and achieve educational fairness."**
>
> — [PM Accelerator](https://www.pmaccelerator.io)

---

**Author:** Mazen  
**Date:** June 2026  
**Dataset:** [Global Weather Repository (Kaggle)](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository)  

---

## Table of Contents

1. [Setup & Data Loading](#1)
2. [Data Cleaning & Preprocessing](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Interactive Visualizations (Plotly)](#4)
5. [Advanced EDA & Anomaly Detection](#5)
6. [Feature Importance Analysis](#6)
7. [Time Series Forecasting — Statistical Models](#7)
8. [Time Series Forecasting — ML Models](#8)
9. [Cross-Validation & Hyperparameter Tuning](#9)
10. [Ensemble Model & Residual Diagnostics](#10)
11. [Climate Analysis](#11)
12. [Environmental Impact — Air Quality](#12)
13. [Spatial & Geographical Analysis](#13)
14. [Conclusions & Key Insights](#14)

<a id='1'></a>

---
## 1. Setup & Data Loading
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from prophet import Prophet
import xgboost as xgb
import pycountry_convert as pc
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='statsmodels')
warnings.filterwarnings('ignore', category=UserWarning, module='prophet')
warnings.filterwarnings('ignore', message='.*cmdstanpy.*')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully.')

In [ ]:
import kagglehub, os

path = kagglehub.dataset_download("nelgiriyewithana/global-weather-repository")
print("Dataset downloaded to:", path)

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print("CSV files found:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f"\nDataset shape: {df.shape}")
print(f"Total records: {df.shape[0]:,}")
print(f"Total features: {df.shape[1]}")

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape}")
print(f"\nColumn names ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)

In [ ]:
df.head(10)

In [ ]:
df.describe()

<a id='2'></a>

---
## 2. Data Cleaning & Preprocessing
---

In [ ]:
# ── Missing value analysis ──
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) > 0:
    print("Columns with missing values:")
    print(missing_df)

    fig, ax = plt.subplots(figsize=(12, max(4, len(missing_df) * 0.4)))
    missing_df['Missing %'].plot(kind='barh', color='coral', ax=ax)
    ax.set_xlabel('Missing Percentage (%)')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset.")

In [ ]:
# ── Imputation: median for numeric, mode for categorical ──
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Remaining missing values: {df.isnull().sum().sum()}")

In [ ]:
# ── Parse datetime ──
date_col = None
for candidate in ['last_updated', 'lastupdated', 'Last Updated', 'LastUpdated', 'date']:
    if candidate in df.columns:
        date_col = candidate
        break
if date_col is None:
    for col in df.columns:
        if 'update' in col.lower() or 'date' in col.lower():
            date_col = col
            break

print(f"Date column identified: '{date_col}'")
df[date_col] = pd.to_datetime(df[date_col])
df['date'] = df[date_col].dt.date
df['year'] = df[date_col].dt.year
df['month'] = df[date_col].dt.month
df['day_of_year'] = df[date_col].dt.dayofyear
df['week'] = df[date_col].dt.isocalendar().week.astype(int)

print(f"Date range: {df[date_col].min()} to {df[date_col].max()}")
print(f"Unique dates: {df[date_col].dt.date.nunique()}")

In [ ]:
# ── Identify key columns dynamically ──
def find_col(df, candidates, substring=None):
    for c in candidates:
        if c in df.columns:
            return c
    if substring:
        for col in df.columns:
            if all(s in col.lower() for s in substring):
                return col
    return None

temp_col = find_col(df, ['temperature_celsius', 'temp_c', 'temperature'], ['temp', 'celsius'])
precip_col = find_col(df, ['precip_mm', 'precipitation_mm', 'precip'], ['precip', 'mm'])
humidity_col = find_col(df, ['humidity', 'relative_humidity'])
wind_col = find_col(df, ['wind_kph', 'wind_speed_kph', 'wind_mph'])
country_col = find_col(df, ['country', 'Country'])
location_col = find_col(df, ['location_name', 'city', 'location', 'name'])
lat_col = find_col(df, ['latitude', 'lat'])
lon_col = find_col(df, ['longitude', 'lon', 'lng'])
aq_cols = [col for col in df.columns if 'air_quality' in col.lower()]

key_numeric = [c for c in [temp_col, precip_col, humidity_col, wind_col] if c is not None]

print(f"Temperature: {temp_col}")
print(f"Precipitation: {precip_col}")
print(f"Humidity: {humidity_col}")
print(f"Wind: {wind_col}")
print(f"Country: {country_col}")
print(f"Location: {location_col}")
print(f"Lat/Lon: {lat_col}, {lon_col}")
print(f"Air quality columns ({len(aq_cols)}): {aq_cols}")

In [ ]:
# ── Outlier detection (IQR) ──
print("Outlier Analysis (IQR Method)")
print("=" * 50)
for col in key_numeric:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_outliers} outliers ({n_outliers/len(df)*100:.2f}%) | Range: [{lower:.1f}, {upper:.1f}]")

fig, axes = plt.subplots(1, len(key_numeric), figsize=(5 * len(key_numeric), 6))
if len(key_numeric) == 1:
    axes = [axes]
for ax, col in zip(axes, key_numeric):
    sns.boxplot(y=df[col], ax=ax, color='skyblue')
    ax.set_title(f'{col}')
plt.suptitle('Outlier Detection - Box Plots', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Winsorize & normalize ──
df_clean = df.copy()
for col in key_numeric:
    lo, hi = df_clean[col].quantile(0.01), df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(lo, hi)

scaler = StandardScaler()
normalized_data = scaler.fit_transform(df_clean[key_numeric])
df_normalized = pd.DataFrame(normalized_data, columns=[f"{c}_norm" for c in key_numeric])

print("Outliers capped at 1st/99th percentiles.")
print("Normalized data:")
print(df_normalized.describe().round(3))

<a id='3'></a>

---
## 3. Exploratory Data Analysis (EDA)
---

In [ ]:
# ── Distribution of key features ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for ax, col, color in zip(axes.flatten(), key_numeric, colors):
    ax.hist(df[col].dropna(), bins=60, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', linestyle='--', lw=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='gray', linestyle=':', lw=1.5, label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'Distribution of {col}', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Distribution of Key Weather Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Temperature trend over time ──
daily_avg_temp = df.groupby('date')[temp_col].mean().reset_index()
daily_avg_temp['date'] = pd.to_datetime(daily_avg_temp['date'])
daily_avg_temp = daily_avg_temp.sort_values('date')

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily_avg_temp['date'], daily_avg_temp[temp_col], color='#e74c3c', alpha=0.5, lw=0.8, label='Daily Avg')
if len(daily_avg_temp) >= 7:
    rolling = daily_avg_temp[temp_col].rolling(window=7, center=True).mean()
    ax.plot(daily_avg_temp['date'], rolling, color='darkred', lw=2.5, label='7-Day Rolling Avg')
if len(daily_avg_temp) >= 30:
    rolling30 = daily_avg_temp[temp_col].rolling(window=30, center=True).mean()
    ax.plot(daily_avg_temp['date'], rolling30, color='black', lw=2, linestyle='--', label='30-Day Rolling Avg')
ax.set_title('Global Average Temperature Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (\u00b0C)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Precipitation trend over time ──
if precip_col:
    daily_avg_precip = df.groupby('date')[precip_col].mean().reset_index()
    daily_avg_precip['date'] = pd.to_datetime(daily_avg_precip['date'])
    daily_avg_precip = daily_avg_precip.sort_values('date')

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(daily_avg_precip['date'], daily_avg_precip[precip_col], color='#3498db', alpha=0.6, width=1)
    if len(daily_avg_precip) >= 7:
        ax.plot(daily_avg_precip['date'],
                daily_avg_precip[precip_col].rolling(window=7, center=True).mean(),
                color='darkblue', lw=2, label='7-Day Rolling Avg')
    ax.set_title('Global Average Precipitation Over Time', fontsize=14)
    ax.set_xlabel('Date')
    ax.set_ylabel('Precipitation (mm)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Monthly temperature patterns ──
monthly_stats = df.groupby('month')[temp_col].agg(['mean', 'std', 'min', 'max']).reset_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(monthly_stats['month'], monthly_stats['mean'], color='coral', alpha=0.7, edgecolor='white')
ax.errorbar(monthly_stats['month'], monthly_stats['mean'], yerr=monthly_stats['std'],
            fmt='none', color='black', capsize=4)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_title('Average Temperature by Month (Global)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (\u00b0C)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ──
corr_cols = list(key_numeric)
for c in aq_cols:
    if c in df.select_dtypes(include=[np.number]).columns:
        corr_cols.append(c)
for c in ['uv_index', 'visibility_km', 'cloud', 'pressure_mb', 'feels_like_celsius']:
    if c in df.columns and c not in corr_cols:
        corr_cols.append(c)

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Correlation Matrix - Weather Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pairwise scatter plots ──
if humidity_col:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    axes[0].scatter(df[temp_col], df[humidity_col], alpha=0.05, s=5, color='teal')
    axes[0].set_title('Temperature vs Humidity')
    axes[0].set_xlabel(temp_col); axes[0].set_ylabel(humidity_col)
    if precip_col:
        axes[1].scatter(df[temp_col], df[precip_col], alpha=0.05, s=5, color='royalblue')
        axes[1].set_title('Temperature vs Precipitation')
        axes[1].set_xlabel(temp_col); axes[1].set_ylabel(precip_col)
    if wind_col:
        axes[2].scatter(df[temp_col], df[wind_col], alpha=0.05, s=5, color='purple')
        axes[2].set_title('Temperature vs Wind Speed')
        axes[2].set_xlabel(temp_col); axes[2].set_ylabel(wind_col)
    plt.suptitle('Pairwise Relationships - Key Weather Variables', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top 15 hottest and coldest countries ──
if country_col:
    country_temp = df.groupby(country_col)[temp_col].mean().sort_values()
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    country_temp.tail(15).plot(kind='barh', ax=axes[0], color='#e74c3c')
    axes[0].set_title('Top 15 Hottest Countries (Avg Temp)', fontsize=13)
    axes[0].set_xlabel('Average Temperature (\u00b0C)')
    country_temp.head(15).plot(kind='barh', ax=axes[1], color='#3498db')
    axes[1].set_title('Top 15 Coldest Countries (Avg Temp)', fontsize=13)
    axes[1].set_xlabel('Average Temperature (\u00b0C)')
    plt.tight_layout()
    plt.show()

<a id='4'></a>

---
## 4. Interactive Visualizations (Plotly)
---

In [ ]:
# ── Interactive temperature time series ──
daily_avg_temp_dt = daily_avg_temp.copy()
daily_avg_temp_dt['rolling_7'] = daily_avg_temp_dt[temp_col].rolling(7, center=True).mean()
daily_avg_temp_dt['rolling_30'] = daily_avg_temp_dt[temp_col].rolling(30, center=True).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_avg_temp_dt['date'], y=daily_avg_temp_dt[temp_col],
                         mode='lines', name='Daily Avg', line=dict(color='#e74c3c', width=1),
                         opacity=0.5))
fig.add_trace(go.Scatter(x=daily_avg_temp_dt['date'], y=daily_avg_temp_dt['rolling_7'],
                         mode='lines', name='7-Day Rolling', line=dict(color='darkred', width=2.5)))
fig.add_trace(go.Scatter(x=daily_avg_temp_dt['date'], y=daily_avg_temp_dt['rolling_30'],
                         mode='lines', name='30-Day Rolling', line=dict(color='black', width=2, dash='dash')))
fig.update_layout(title='Interactive: Global Average Temperature Over Time',
                  xaxis_title='Date', yaxis_title='Temperature (\u00b0C)',
                  template='plotly_white', hovermode='x unified', height=500)
fig.show()

In [ ]:
# ── Interactive global temperature map ──
if lat_col and lon_col and location_col:
    loc_avg = df.groupby(location_col).agg({
        lat_col: 'first', lon_col: 'first', temp_col: 'mean',
        country_col: 'first' if country_col else temp_col
    }).reset_index()

    fig = px.scatter_geo(loc_avg, lat=lat_col, lon=lon_col, color=temp_col,
                         hover_name=location_col, color_continuous_scale='RdYlBu_r',
                         title='Interactive: Global Temperature Distribution',
                         size_max=8, opacity=0.7,
                         labels={temp_col: 'Avg Temp (\u00b0C)'})
    fig.update_layout(height=600, template='plotly_white')
    fig.update_geos(showcountries=True, countrycolor='lightgray')
    fig.show()

In [ ]:
# ── Interactive monthly temperature box plot ──
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['month_name'] = df['month'].map(month_map)

sample = df.sample(min(50000, len(df)), random_state=42)
fig = px.box(sample, x='month_name', y=temp_col,
             category_orders={'month_name': list(month_map.values())},
             color='month_name', title='Interactive: Monthly Temperature Distribution',
             labels={temp_col: 'Temperature (\u00b0C)', 'month_name': 'Month'})
fig.update_layout(showlegend=False, template='plotly_white', height=500)
fig.show()

In [ ]:
# ── Interactive correlation heatmap ──
fig = px.imshow(corr_matrix, text_auto='.2f', color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, aspect='auto',
                title='Interactive: Correlation Matrix')
fig.update_layout(height=700, width=900, template='plotly_white')
fig.show()

In [ ]:
# ── Interactive top countries bar chart ──
if country_col:
    top20 = df.groupby(country_col)[temp_col].mean().nlargest(20).reset_index()
    fig = px.bar(top20, x=temp_col, y=country_col, orientation='h',
                 color=temp_col, color_continuous_scale='Reds',
                 title='Interactive: Top 20 Hottest Countries',
                 labels={temp_col: 'Avg Temperature (\u00b0C)', country_col: 'Country'})
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, template='plotly_white', height=500)
    fig.show()

<a id='5'></a>

---
## 5. Advanced EDA & Anomaly Detection
---

In [ ]:
# ── Isolation Forest anomaly detection ──
anomaly_features = [c for c in [temp_col, humidity_col, wind_col, precip_col] if c is not None]
anomaly_data = df[anomaly_features].dropna()

iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
anomaly_labels = iso_forest.fit_predict(anomaly_data)

df_anomaly = df.loc[anomaly_data.index].copy()
df_anomaly['is_anomaly'] = anomaly_labels == -1

n_anomalies = df_anomaly['is_anomaly'].sum()
print(f"Anomalies detected: {n_anomalies} ({n_anomalies/len(df_anomaly)*100:.2f}%)")

normal = df_anomaly[~df_anomaly['is_anomaly']]
anomalies = df_anomaly[df_anomaly['is_anomaly']]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].scatter(normal[temp_col], normal[humidity_col], alpha=0.03, s=3, color='steelblue', label='Normal')
axes[0].scatter(anomalies[temp_col], anomalies[humidity_col], alpha=0.3, s=10, color='red', label='Anomaly')
axes[0].set_title('Temperature vs Humidity', fontsize=13)
axes[0].set_xlabel(temp_col); axes[0].set_ylabel(humidity_col); axes[0].legend()

if wind_col and precip_col:
    axes[1].scatter(normal[wind_col], normal[precip_col], alpha=0.03, s=3, color='steelblue', label='Normal')
    axes[1].scatter(anomalies[wind_col], anomalies[precip_col], alpha=0.3, s=10, color='red', label='Anomaly')
    axes[1].set_title('Wind vs Precipitation', fontsize=13)
    axes[1].set_xlabel(wind_col); axes[1].set_ylabel(precip_col); axes[1].legend()

plt.suptitle('Isolation Forest Anomaly Detection (5% contamination)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Z-score anomaly analysis ──
print("Z-Score Anomaly Analysis (|z| > 3)")
print("=" * 50)
for col in key_numeric:
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    n_extreme = (z_scores > 3).sum()
    print(f"{col}: {n_extreme} extreme values ({n_extreme/len(df)*100:.3f}%)")

print("\nAnomaly vs Normal - Feature Comparison")
print("=" * 60)
comparison = pd.DataFrame({
    'Normal Mean': normal[anomaly_features].mean(),
    'Anomaly Mean': anomalies[anomaly_features].mean(),
    'Normal Std': normal[anomaly_features].std(),
    'Anomaly Std': anomalies[anomaly_features].std()
})
print(comparison.round(2))

<a id='6'></a>

---
## 6. Feature Importance Analysis

Three methods: Random Forest, Permutation, and Correlation-based.

---

In [ ]:
# ── Prepare features ──
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in [temp_col, 'year', 'month', 'day_of_year', 'week']
                and 'fahrenheit' not in c.lower()
                and 'feels_like' not in c.lower()
                and 'epoch' not in c.lower()]
feature_cols = [c for c in feature_cols if df[c].notna().sum() > len(df) * 0.5]

X_fi = df[feature_cols].dropna()
y_fi = df.loc[X_fi.index, temp_col]

if len(X_fi) > 50000:
    idx = np.random.RandomState(42).choice(X_fi.index, 50000, replace=False)
    X_fi, y_fi = X_fi.loc[idx], y_fi.loc[idx]

print(f"Feature importance: {X_fi.shape[0]} samples, {X_fi.shape[1]} features")

In [ ]:
# ── Three methods (all use the same train/test split for consistency) ──
X_tr, X_te, y_tr, y_te = train_test_split(X_fi, y_fi, test_size=0.2, random_state=42)

# Method 1: Random Forest feature importance (fit on train set only)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_tr, y_tr)
rf_importance = pd.Series(rf_model.feature_importances_, index=X_fi.columns).sort_values(ascending=False)

# Method 2: Permutation Importance (evaluated on held-out test set)
perm_imp = permutation_importance(rf_model, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
perm_importance = pd.Series(perm_imp.importances_mean, index=X_fi.columns).sort_values(ascending=False)

# Method 3: Correlation-based (dataset-level, no train/test needed)
corr_importance = df[feature_cols + [temp_col]].corr()[temp_col].drop(temp_col).abs().sort_values(ascending=False)

print(f"RF model R² on test set: {rf_model.score(X_te, y_te):.4f}")
print("\nFeature Importance Rankings (Top 10)")
print("=" * 60)
print(pd.DataFrame({'RF': rf_importance.head(10), 'Permutation': perm_importance.head(10),
                     'Correlation': corr_importance.head(10)}).round(4))

In [ ]:
# ── Visualize all 3 methods ──
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
top_n = min(15, len(rf_importance))

rf_importance.head(top_n).sort_values().plot(kind='barh', ax=axes[0], color='#27ae60')
axes[0].set_title('Random Forest\nFeature Importance', fontsize=13)
axes[0].set_xlabel('Importance')

perm_importance.head(top_n).sort_values().plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Permutation\nImportance', fontsize=13)
axes[1].set_xlabel('Mean Decrease in R\u00b2')

corr_importance.head(top_n).sort_values().plot(kind='barh', ax=axes[2], color='#8e44ad')
axes[2].set_title('Correlation-Based\nImportance', fontsize=13)
axes[2].set_xlabel('|Correlation| with Temperature')

plt.suptitle('Feature Importance Analysis - Predicting Temperature', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<a id='7'></a>

---
## 7. Time Series Forecasting - Statistical Models

Daily global average temperature using the `last_updated` feature. Models: ARIMA, SARIMA, Holt-Winters, Prophet.

---

In [ ]:
# ── Prepare time series ──
ts = df.groupby(df[date_col].dt.date)[temp_col].mean()
ts.index = pd.to_datetime(ts.index)
ts = ts.sort_index().asfreq('D').interpolate(method='linear')
ts.name = 'temperature'

split_idx = int(len(ts) * 0.8)
train_ts, test_ts = ts[:split_idx], ts[split_idx:]

print(f"Time series: {len(ts)} days ({ts.index.min().date()} to {ts.index.max().date()})")
print(f"Train: {len(train_ts)} days | Test: {len(test_ts)} days")
print(f"Mean: {ts.mean():.2f}\u00b0C, Std: {ts.std():.2f}\u00b0C")

In [ ]:
# ── Stationarity test ──
adf_result = adfuller(train_ts.dropna())
print("Augmented Dickey-Fuller Test")
print(f"  Test Statistic: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.6f}")
print(f"  Stationary: {'Yes' if adf_result[1] < 0.05 else 'No'}")

# Seasonal decomposition
if len(train_ts) >= 14:
    period = min(7, len(train_ts) // 3)
    decomp = seasonal_decompose(train_ts, model='additive', period=period)
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    decomp.observed.plot(ax=axes[0], title='Observed', color='steelblue')
    decomp.trend.plot(ax=axes[1], title='Trend', color='#e74c3c')
    decomp.seasonal.plot(ax=axes[2], title='Seasonal', color='#2ecc71')
    decomp.resid.plot(ax=axes[3], title='Residual', color='#9b59b6')
    plt.suptitle('Time Series Decomposition', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── ACF / PACF ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
max_lags = min(40, len(train_ts) // 2 - 1)
plot_acf(train_ts.dropna(), lags=max_lags, ax=axes[0], title='Autocorrelation (ACF)')
plot_pacf(train_ts.dropna(), lags=max_lags, ax=axes[1], title='Partial Autocorrelation (PACF)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluation helper ──
def evaluate_forecast(actual, predicted, model_name):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual.replace(0, np.nan))) * 100
    r2 = r2_score(actual, predicted)
    return {'Model': model_name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
            'MAPE (%)': round(mape, 4), 'R\u00b2': round(r2, 4)}

results = []
forecasts = {}

In [ ]:
# ── Model 1: ARIMA (auto-selected order via pmdarima) ──
from pmdarima import auto_arima

print("Running auto_arima to find optimal order...")
auto_model = auto_arima(train_ts, seasonal=False, stepwise=True, suppress_warnings=True,
                         max_p=5, max_q=5, max_d=2, information_criterion='aic')
best_order = auto_model.order
print(f"  Best order: ARIMA{best_order} (AIC: {auto_model.aic():.2f})")

try:
    arima_fit = ARIMA(train_ts, order=best_order).fit()
    arima_pred = arima_fit.forecast(steps=len(test_ts))
    arima_pred.index = test_ts.index
    results.append(evaluate_forecast(test_ts, arima_pred, f'ARIMA{best_order}'))
    forecasts['ARIMA'] = arima_pred
    print(f"  RMSE: {results[-1]['RMSE']}")
except Exception as e:
    print(f"  Auto-selected order failed ({e}), trying ARIMA(2,1,1)...")
    arima_fit = ARIMA(train_ts, order=(2, 1, 1)).fit()
    arima_pred = arima_fit.forecast(steps=len(test_ts))
    arima_pred.index = test_ts.index
    results.append(evaluate_forecast(test_ts, arima_pred, 'ARIMA(2,1,1)'))
    forecasts['ARIMA'] = arima_pred
    print(f"  Fallback RMSE: {results[-1]['RMSE']}")

In [ ]:
# ── Model 2: SARIMA ──
print("Training SARIMA(2,1,1)(1,1,1,7)...")
try:
    sarima_model = SARIMAX(train_ts, order=(2, 1, 1), seasonal_order=(1, 1, 1, 7),
                           enforce_stationarity=False, enforce_invertibility=False)
    sarima_fit = sarima_model.fit(disp=False, maxiter=200)
    sarima_pred = sarima_fit.forecast(steps=len(test_ts))
    sarima_pred.index = test_ts.index
    results.append(evaluate_forecast(test_ts, sarima_pred, 'SARIMA(2,1,1)(1,1,1,7)'))
    forecasts['SARIMA'] = sarima_pred
    print(f"  RMSE: {results[-1]['RMSE']}")
except Exception as e:
    print(f"  Failed: {e}")
    try:
        sarima_model = SARIMAX(train_ts, order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
                               enforce_stationarity=False, enforce_invertibility=False)
        sarima_fit = sarima_model.fit(disp=False, maxiter=200)
        sarima_pred = sarima_fit.forecast(steps=len(test_ts))
        sarima_pred.index = test_ts.index
        results.append(evaluate_forecast(test_ts, sarima_pred, 'SARIMA(1,1,1)(1,0,1,7)'))
        forecasts['SARIMA'] = sarima_pred
        print(f"  Fallback RMSE: {results[-1]['RMSE']}")
    except Exception as e2:
        print(f"  Fallback also failed: {e2}")

In [ ]:
# ── Model 3: Holt-Winters ──
print("Training Holt-Winters...")
try:
    period = min(7, len(train_ts) // 3)
    if len(train_ts) >= 2 * period:
        hw_fit = ExponentialSmoothing(train_ts, trend='add', seasonal='add',
                                      seasonal_periods=period).fit(optimized=True)
        hw_pred = hw_fit.forecast(steps=len(test_ts))
        hw_pred.index = test_ts.index
        results.append(evaluate_forecast(test_ts, hw_pred, 'Holt-Winters'))
        forecasts['Holt-Winters'] = hw_pred
        print(f"  RMSE: {results[-1]['RMSE']}")
    else:
        hw_fit = ExponentialSmoothing(train_ts, trend='add', seasonal=None).fit(optimized=True)
        hw_pred = hw_fit.forecast(steps=len(test_ts))
        hw_pred.index = test_ts.index
        results.append(evaluate_forecast(test_ts, hw_pred, 'Holt (No Seasonal)'))
        forecasts['Holt-Winters'] = hw_pred
        print(f"  RMSE: {results[-1]['RMSE']}")
except Exception as e:
    print(f"  Failed: {e}")

In [ ]:
# ── Model 4: Prophet ──
print("Training Prophet...")
try:
    prophet_df = pd.DataFrame({'ds': train_ts.index, 'y': train_ts.values})
    prophet_model = Prophet(daily_seasonality=True, weekly_seasonality=True,
                            yearly_seasonality=True if len(train_ts) > 365 else False,
                            changepoint_prior_scale=0.05)
    prophet_model.fit(prophet_df)

    future = prophet_model.make_future_dataframe(periods=len(test_ts))
    prophet_forecast = prophet_model.predict(future)

    prophet_pred = prophet_forecast.set_index('ds').loc[test_ts.index, 'yhat']
    results.append(evaluate_forecast(test_ts, prophet_pred, 'Prophet'))
    forecasts['Prophet'] = prophet_pred
    print(f"  RMSE: {results[-1]['RMSE']}")

    # Prophet components
    fig = prophet_model.plot_components(prophet_forecast)
    plt.suptitle('Prophet - Trend & Seasonal Components', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"  Prophet failed: {e}")

<a id='8'></a>

---
## 8. Time Series Forecasting - ML Models

Ridge Regression, Random Forest, XGBoost, Gradient Boosting with engineered lag features.

---

In [ ]:
# ── Feature engineering ──
def create_lag_features(series, n_lags=14):
    df_lag = pd.DataFrame({'target': series})
    for i in range(1, n_lags + 1):
        df_lag[f'lag_{i}'] = series.shift(i)
    df_lag['rolling_mean_7'] = series.rolling(7).mean()
    df_lag['rolling_std_7'] = series.rolling(7).std()
    df_lag['rolling_mean_14'] = series.rolling(14).mean()
    df_lag['rolling_min_7'] = series.rolling(7).min()
    df_lag['rolling_max_7'] = series.rolling(7).max()
    df_lag['day_of_year'] = series.index.dayofyear
    df_lag['month'] = series.index.month
    df_lag['day_of_week'] = series.index.dayofweek
    df_lag['sin_day'] = np.sin(2 * np.pi * series.index.dayofyear / 365.25)
    df_lag['cos_day'] = np.cos(2 * np.pi * series.index.dayofyear / 365.25)
    df_lag = df_lag.dropna()
    return df_lag

lag_df = create_lag_features(ts, n_lags=14)
lag_features = [c for c in lag_df.columns if c != 'target']

train_lag = lag_df[lag_df.index < test_ts.index[0]]
test_lag = lag_df[lag_df.index >= test_ts.index[0]]

print(f"Lag features: {len(lag_features)}")
print(f"Train: {len(train_lag)} | Test: {len(test_lag)}")
print(f"Features: {lag_features}")

In [ ]:
# ── Model 5: Ridge Regression ──
print("Training Ridge Regression...")
if len(test_lag) > 0:
    ridge = Ridge(alpha=1.0)
    ridge.fit(train_lag[lag_features], train_lag['target'])
    ridge_pred = pd.Series(ridge.predict(test_lag[lag_features]), index=test_lag.index)
    test_actual = test_lag['target']
    results.append(evaluate_forecast(test_actual, ridge_pred, 'Ridge Regression'))
    forecasts['Ridge'] = ridge_pred
    print(f"  RMSE: {results[-1]['RMSE']}")

In [ ]:
# ── Model 6: Random Forest ──
print("Training Random Forest...")
if len(test_lag) > 0:
    rf_ts = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    rf_ts.fit(train_lag[lag_features], train_lag['target'])
    rf_pred = pd.Series(rf_ts.predict(test_lag[lag_features]), index=test_lag.index)
    results.append(evaluate_forecast(test_actual, rf_pred, 'Random Forest'))
    forecasts['Random Forest'] = rf_pred
    print(f"  RMSE: {results[-1]['RMSE']}")

In [ ]:
# ── Model 7: XGBoost ──
print("Training XGBoost...")
if len(test_lag) > 0:
    xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                  random_state=42, n_jobs=-1, verbosity=0)
    xgb_model.fit(train_lag[lag_features], train_lag['target'])
    xgb_pred = pd.Series(xgb_model.predict(test_lag[lag_features]), index=test_lag.index)
    results.append(evaluate_forecast(test_actual, xgb_pred, 'XGBoost'))
    forecasts['XGBoost'] = xgb_pred
    print(f"  RMSE: {results[-1]['RMSE']}")

In [ ]:
# ── Model 8: Gradient Boosting ──
print("Training Gradient Boosting...")
if len(test_lag) > 0:
    gb_model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
    gb_model.fit(train_lag[lag_features], train_lag['target'])
    gb_pred = pd.Series(gb_model.predict(test_lag[lag_features]), index=test_lag.index)
    results.append(evaluate_forecast(test_actual, gb_pred, 'Gradient Boosting'))
    forecasts['Gradient Boosting'] = gb_pred
    print(f"  RMSE: {results[-1]['RMSE']}")

In [ ]:
# ── Model comparison table ──
results_df = pd.DataFrame(results).sort_values('RMSE')
print("\n" + "=" * 75)
print("MODEL COMPARISON - All 8 Models")
print("=" * 75)
print(results_df.to_string(index=False))
print(f"\nBest model: {results_df.iloc[0]['Model']} (RMSE: {results_df.iloc[0]['RMSE']})")

In [ ]:
# ── Plotly interactive model comparison ──
fig = make_subplots(rows=1, cols=2, subplot_titles=['RMSE (lower = better)', 'MAE (lower = better)'])

fig.add_trace(go.Bar(y=results_df['Model'], x=results_df['RMSE'], orientation='h',
                     marker_color=px.colors.qualitative.Set2[:len(results_df)], name='RMSE'), row=1, col=1)
fig.add_trace(go.Bar(y=results_df['Model'], x=results_df['MAE'], orientation='h',
                     marker_color=px.colors.qualitative.Set2[:len(results_df)], name='MAE'), row=1, col=2)
fig.update_layout(title='Model Performance Comparison', showlegend=False,
                  template='plotly_white', height=450)
fig.show()

In [ ]:
# ── Plotly interactive forecast visualization ──
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_ts.index, y=train_ts, mode='lines', name='Training',
                         line=dict(color='gray', width=1), opacity=0.5))
fig.add_trace(go.Scatter(x=test_ts.index, y=test_ts, mode='lines', name='Actual (Test)',
                         line=dict(color='black', width=2.5)))

model_colors = {'ARIMA': '#e74c3c', 'SARIMA': '#c0392b', 'Holt-Winters': '#3498db',
                'Prophet': '#f39c12', 'Ridge': '#2ecc71', 'Random Forest': '#9b59b6',
                'XGBoost': '#e67e22', 'Gradient Boosting': '#1abc9c'}
for name, pred in forecasts.items():
    fig.add_trace(go.Scatter(x=pred.index, y=pred, mode='lines', name=name,
                             line=dict(color=model_colors.get(name, 'gray'), width=1.5, dash='dash')))

fig.add_vline(x=test_ts.index[0], line_dash='dot', line_color='red', opacity=0.5)
fig.update_layout(title='Interactive: All Model Forecasts vs Actual',
                  xaxis_title='Date', yaxis_title='Temperature (\u00b0C)',
                  template='plotly_white', hovermode='x unified', height=550)
fig.show()

<a id='9'></a>

---
## 9. Cross-Validation & Hyperparameter Tuning

Use `TimeSeriesSplit` for proper temporal cross-validation and `GridSearchCV` for hyperparameter optimization.

---

In [ ]:
# ── TimeSeriesSplit cross-validation for all ML models ──
tscv = TimeSeriesSplit(n_splits=5)

cv_models = {
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

cv_results = {}
X_all = train_lag[lag_features].values
y_all = train_lag['target'].values

print("TimeSeriesSplit Cross-Validation (5-fold)")
print("=" * 60)

for name, model in cv_models.items():
    fold_rmses = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
        X_cv_train, X_cv_val = X_all[train_idx], X_all[val_idx]
        y_cv_train, y_cv_val = y_all[train_idx], y_all[val_idx]
        model.fit(X_cv_train, y_cv_train)
        preds = model.predict(X_cv_val)
        fold_rmse = np.sqrt(mean_squared_error(y_cv_val, preds))
        fold_rmses.append(fold_rmse)
    cv_results[name] = fold_rmses
    print(f"{name:20s} | Mean RMSE: {np.mean(fold_rmses):.4f} +/- {np.std(fold_rmses):.4f} | Folds: {[f'{r:.4f}' for r in fold_rmses]}")

# Visualize CV results
fig, ax = plt.subplots(figsize=(12, 6))
cv_df = pd.DataFrame(cv_results)
cv_df.index = [f'Fold {i+1}' for i in range(5)]
cv_df.plot(kind='bar', ax=ax, width=0.7, alpha=0.8)
ax.set_title('TimeSeriesSplit Cross-Validation RMSE by Fold', fontsize=14)
ax.set_ylabel('RMSE')
ax.set_xlabel('Fold')
ax.legend(title='Model')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Hyperparameter tuning: XGBoost with TimeSeriesSplit ──
print("Hyperparameter Tuning: XGBoost (GridSearchCV + TimeSeriesSplit)")
print("=" * 60)

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2]
}

xgb_grid = GridSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    xgb_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=0
)
xgb_grid.fit(X_all, y_all)

print(f"Best params: {xgb_grid.best_params_}")
print(f"Best CV RMSE: {-xgb_grid.best_score_:.4f}")

# Retrain with best params and evaluate
xgb_tuned = xgb_grid.best_estimator_
xgb_tuned.fit(train_lag[lag_features], train_lag['target'])
xgb_tuned_pred = pd.Series(xgb_tuned.predict(test_lag[lag_features]), index=test_lag.index)
tuned_eval = evaluate_forecast(test_actual, xgb_tuned_pred, 'XGBoost (Tuned)')
results.append(tuned_eval)
forecasts['XGBoost (Tuned)'] = xgb_tuned_pred
print(f"Test RMSE: {tuned_eval['RMSE']}")

In [ ]:
# ── Hyperparameter tuning: Random Forest with TimeSeriesSplit ──
print("\nHyperparameter Tuning: Random Forest (GridSearchCV + TimeSeriesSplit)")
print("=" * 60)

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_all, y_all)

print(f"Best params: {rf_grid.best_params_}")
print(f"Best CV RMSE: {-rf_grid.best_score_:.4f}")

rf_tuned = rf_grid.best_estimator_
rf_tuned.fit(train_lag[lag_features], train_lag['target'])
rf_tuned_pred = pd.Series(rf_tuned.predict(test_lag[lag_features]), index=test_lag.index)
tuned_rf_eval = evaluate_forecast(test_actual, rf_tuned_pred, 'Random Forest (Tuned)')
results.append(tuned_rf_eval)
forecasts['RF (Tuned)'] = rf_tuned_pred
print(f"Test RMSE: {tuned_rf_eval['RMSE']}")

In [ ]:
# ── Updated model comparison ──
results_df = pd.DataFrame(results).sort_values('RMSE')
print("\n" + "=" * 75)
print("UPDATED MODEL COMPARISON (with Tuned Models)")
print("=" * 75)
print(results_df.to_string(index=False))

<a id='10'></a>

---
## 10. Ensemble Model

Combine top-performing models using simple average and inverse-RMSE weighted ensemble.

---

In [ ]:
# ── Ensemble: combine ML models ──
ml_keys = ['Ridge', 'Random Forest', 'XGBoost', 'Gradient Boosting', 'XGBoost (Tuned)', 'RF (Tuned)']
ml_forecasts = {k: v for k, v in forecasts.items() if k in ml_keys}

if len(ml_forecasts) >= 2:
    common_idx = ml_forecasts[list(ml_forecasts.keys())[0]].index
    for v in ml_forecasts.values():
        common_idx = common_idx.intersection(v.index)

    actual_common = test_actual.loc[common_idx]

    # Simple average
    ensemble_simple = sum(v.loc[common_idx] for v in ml_forecasts.values()) / len(ml_forecasts)

    # Weighted by inverse RMSE
    rmse_dict = {k: np.sqrt(mean_squared_error(actual_common, v.loc[common_idx])) for k, v in ml_forecasts.items()}
    inv_rmse = {k: 1.0 / v for k, v in rmse_dict.items()}
    total_inv = sum(inv_rmse.values())
    weights = {k: v / total_inv for k, v in inv_rmse.items()}

    ensemble_weighted = sum(weights[k] * v.loc[common_idx] for k, v in ml_forecasts.items())

    simple_eval = evaluate_forecast(actual_common, ensemble_simple, 'Ensemble (Simple Avg)')
    weighted_eval = evaluate_forecast(actual_common, ensemble_weighted, 'Ensemble (Weighted)')
    results.append(simple_eval)
    results.append(weighted_eval)

    print("Ensemble Weights:")
    for name, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f"  {name}: {w:.3f}")
    print(f"\nSimple Ensemble RMSE: {simple_eval['RMSE']}")
    print(f"Weighted Ensemble RMSE: {weighted_eval['RMSE']}")

    # Final ranking
    results_df = pd.DataFrame(results).sort_values('RMSE')
    print("\n" + "=" * 75)
    print("FINAL MODEL COMPARISON (12 Models)")
    print("=" * 75)
    print(results_df.to_string(index=False))
    print(f"\nBest overall: {results_df.iloc[0]['Model']} (RMSE: {results_df.iloc[0]['RMSE']})")

In [ ]:
# ── Interactive ensemble visualization with residual-based prediction intervals ──
if len(ml_forecasts) >= 2:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=actual_common.index, y=actual_common, mode='lines',
                             name='Actual', line=dict(color='black', width=2.5)))
    fig.add_trace(go.Scatter(x=common_idx, y=ensemble_simple, mode='lines',
                             name=f'Simple Ensemble (RMSE: {simple_eval["RMSE"]})',
                             line=dict(color='#e74c3c', width=2, dash='dash')))
    fig.add_trace(go.Scatter(x=common_idx, y=ensemble_weighted, mode='lines',
                             name=f'Weighted Ensemble (RMSE: {weighted_eval["RMSE"]})',
                             line=dict(color='#3498db', width=2, dash='dash')))

    # Residual-based 95% prediction interval (proper statistical method)
    residuals = actual_common - ensemble_weighted
    residual_std = residuals.std()
    ci_upper = ensemble_weighted + 1.96 * residual_std
    ci_lower = ensemble_weighted - 1.96 * residual_std

    fig.add_trace(go.Scatter(x=list(common_idx) + list(common_idx[::-1]),
                             y=list(ci_upper) + list(ci_lower[::-1]),
                             fill='toself', fillcolor='rgba(52,152,219,0.1)',
                             line=dict(color='rgba(255,255,255,0)'),
                             name=f'95% PI (residual σ={residual_std:.2f}°C)'))
    fig.update_layout(title='Interactive: Ensemble Forecast vs Actual (with Prediction Interval)',
                      xaxis_title='Date', yaxis_title='Temperature (°C)',
                      template='plotly_white', hovermode='x unified', height=550)
    fig.show()

### Residual Diagnostics

Validating that the ensemble model's residuals are well-behaved: normally distributed, zero-mean, and free of autocorrelation.

In [ ]:
# ── Residual diagnostics for the weighted ensemble ──
if len(ml_forecasts) >= 2:
    residuals = actual_common - ensemble_weighted

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Residual distribution with normality test
    axes[0, 0].hist(residuals, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
    x_range = np.linspace(residuals.min(), residuals.max(), 100)
    axes[0, 0].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
                    'r-', lw=2, label='Normal fit')
    axes[0, 0].axvline(0, color='black', linestyle='--', lw=1)
    axes[0, 0].set_title('Residual Distribution', fontsize=13)
    axes[0, 0].set_xlabel('Residual (°C)')
    axes[0, 0].legend()

    # 2. QQ plot
    stats.probplot(residuals, dist="norm", plot=axes[0, 1])
    axes[0, 1].set_title('Q-Q Plot (Normal)', fontsize=13)
    axes[0, 1].get_lines()[0].set_color('steelblue')
    axes[0, 1].get_lines()[1].set_color('red')

    # 3. Residuals over time
    axes[1, 0].plot(residuals.index, residuals, color='steelblue', alpha=0.7, lw=1)
    axes[1, 0].axhline(0, color='black', linestyle='--', lw=1)
    axes[1, 0].fill_between(residuals.index, -1.96 * residuals.std(), 1.96 * residuals.std(),
                             alpha=0.1, color='red', label='±1.96σ')
    axes[1, 0].set_title('Residuals Over Time', fontsize=13)
    axes[1, 0].set_xlabel('Date')
    axes[1, 0].set_ylabel('Residual (°C)')
    axes[1, 0].legend()

    # 4. Residual ACF
    max_acf_lags = min(30, len(residuals) // 2 - 1)
    if max_acf_lags > 1:
        plot_acf(residuals.dropna(), lags=max_acf_lags, ax=axes[1, 1], title='Residual ACF')
    else:
        axes[1, 1].text(0.5, 0.5, 'Insufficient data for ACF', ha='center', va='center', transform=axes[1, 1].transAxes)

    plt.suptitle('Weighted Ensemble — Residual Diagnostics', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()

    # Statistical tests
    shapiro_stat, shapiro_p = stats.shapiro(residuals[:min(5000, len(residuals))])
    print("Residual Diagnostics Summary")
    print("=" * 50)
    print(f"  Mean:     {residuals.mean():.4f}°C (ideal: 0)")
    print(f"  Std:      {residuals.std():.4f}°C")
    print(f"  Skewness: {residuals.skew():.4f} (ideal: 0)")
    print(f"  Kurtosis: {residuals.kurtosis():.4f} (ideal: 0)")
    print(f"  Shapiro-Wilk p-value: {shapiro_p:.4f} ({'Normal' if shapiro_p > 0.05 else 'Non-normal'})")

<a id='11'></a>

---
## 11. Climate Analysis

Long-term climate patterns and variations across regions, climate zones, and hemispheres.

---

In [ ]:
# ── Climate zones based on latitude ──
if lat_col:
    def classify_climate_zone(lat):
        abs_lat = abs(lat)
        if abs_lat <= 23.5: return 'Tropical'
        elif abs_lat <= 35: return 'Subtropical'
        elif abs_lat <= 55: return 'Temperate'
        elif abs_lat <= 66.5: return 'Subarctic'
        else: return 'Polar'

    df['climate_zone'] = df[lat_col].apply(classify_climate_zone)
    df['hemisphere'] = df[lat_col].apply(lambda x: 'Northern' if x >= 0 else 'Southern')
    print("Climate Zone Distribution:")
    print(df['climate_zone'].value_counts())

In [ ]:
# ── Temperature by climate zone (interactive) ──
if lat_col:
    zone_monthly = df.groupby(['climate_zone', 'month'])[temp_col].mean().reset_index()
    zone_monthly['month_name'] = zone_monthly['month'].map(month_map)

    fig = px.line(zone_monthly, x='month_name', y=temp_col, color='climate_zone',
                  category_orders={'month_name': list(month_map.values()),
                                   'climate_zone': ['Tropical','Subtropical','Temperate','Subarctic','Polar']},
                  markers=True, title='Interactive: Temperature by Climate Zone & Month',
                  labels={temp_col: 'Avg Temperature (\u00b0C)', 'month_name': 'Month', 'climate_zone': 'Climate Zone'})
    fig.update_layout(template='plotly_white', height=500)
    fig.show()

In [ ]:
# ── Hemisphere comparison ──
if lat_col:
    hemi_monthly = df.groupby(['hemisphere', 'month'])[temp_col].mean().reset_index()
    hemi_monthly['month_name'] = hemi_monthly['month'].map(month_map)

    fig = px.line(hemi_monthly, x='month_name', y=temp_col, color='hemisphere',
                  category_orders={'month_name': list(month_map.values())},
                  markers=True, title='Northern vs Southern Hemisphere - Seasonal Patterns',
                  labels={temp_col: 'Avg Temperature (\u00b0C)', 'month_name': 'Month'})
    fig.update_layout(template='plotly_white', height=450)
    fig.show()

In [ ]:
# ── Temperature variability by climate zone ──
if lat_col:
    zone_order = ['Tropical', 'Subtropical', 'Temperate', 'Subarctic', 'Polar']
    colors_zone = {'Tropical':'#e74c3c', 'Subtropical':'#e67e22', 'Temperate':'#2ecc71',
                   'Subarctic':'#3498db', 'Polar':'#9b59b6'}

    fig, ax = plt.subplots(figsize=(14, 7))
    zone_data = [df[df['climate_zone'] == z][temp_col].dropna() for z in zone_order if z in df['climate_zone'].values]
    zone_labels = [z for z in zone_order if z in df['climate_zone'].values]
    bp = ax.boxplot(zone_data, labels=zone_labels, patch_artist=True, showfliers=False)
    for patch, zone in zip(bp['boxes'], zone_labels):
        patch.set_facecolor(colors_zone.get(zone, 'gray'))
        patch.set_alpha(0.6)
    ax.set_title('Temperature Distribution by Climate Zone', fontsize=14)
    ax.set_ylabel('Temperature (\u00b0C)')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Climate zone summary statistics ──
if lat_col:
    agg_dict = {'avg_temp': (temp_col, 'mean'), 'temp_std': (temp_col, 'std'), 'n_records': (temp_col, 'count')}
    if precip_col: agg_dict['avg_precip'] = (precip_col, 'mean')
    if humidity_col: agg_dict['avg_humidity'] = (humidity_col, 'mean')

    climate_stats = df.groupby('climate_zone').agg(**agg_dict).round(2)
    print("Climate Zone Summary")
    print("=" * 60)
    print(climate_stats)

<a id='12'></a>

---
## 12. Environmental Impact - Air Quality Analysis

Correlation between air quality indices and weather parameters.

---

In [ ]:
# ── Air quality overview ──
if len(aq_cols) > 0:
    print(f"Air Quality Features ({len(aq_cols)}):")
    for col in aq_cols:
        print(f"  {col}: mean={df[col].mean():.2f}, std={df[col].std():.2f}")

    n_aq = min(len(aq_cols), 6)
    n_rows = (n_aq + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5 * n_rows))
    axes_flat = axes.flatten() if n_rows > 1 else (axes if n_aq > 1 else [axes])
    for i, col in enumerate(aq_cols[:n_aq]):
        axes_flat[i].hist(df[col].dropna(), bins=50, color='#1abc9c', alpha=0.7, edgecolor='white')
        axes_flat[i].set_title(col.replace('air_quality_', 'AQ: '), fontsize=11)
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.suptitle('Air Quality Metrics Distribution', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No air quality columns found.")

In [ ]:
# ── AQ vs weather correlation heatmap ──
if len(aq_cols) > 0:
    weather_params = [c for c in [temp_col, humidity_col, wind_col, precip_col] if c]
    for c in ['pressure_mb', 'visibility_km', 'uv_index', 'cloud']:
        if c in df.columns: weather_params.append(c)

    aq_weather_corr = df[aq_cols + weather_params].corr().loc[aq_cols, weather_params]

    # Interactive heatmap
    fig = px.imshow(aq_weather_corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                    zmin=-1, zmax=1, aspect='auto',
                    title='Interactive: Air Quality vs Weather Correlation')
    fig.update_layout(height=500, template='plotly_white')
    fig.show()

    print("\nStrongest Correlations (|r| > 0.3):")
    for aq in aq_cols:
        for wp in weather_params:
            r = aq_weather_corr.loc[aq, wp]
            if abs(r) > 0.3:
                print(f"  {aq.replace('air_quality_','')} <-> {wp}: {r:.3f}")

In [ ]:
# ── AQ by climate zone ──
if len(aq_cols) > 0 and lat_col:
    aq_main = None
    for c in aq_cols:
        if 'pm2' in c.lower() or 'epa' in c.lower():
            aq_main = c; break
    if not aq_main: aq_main = aq_cols[0]

    zone_aq = df.groupby('climate_zone')[aq_main].mean().reindex(zone_order).dropna().reset_index()
    fig = px.bar(zone_aq, x='climate_zone', y=aq_main, color='climate_zone',
                 title=f'Average {aq_main.replace("air_quality_","")} by Climate Zone',
                 color_discrete_map=colors_zone)
    fig.update_layout(template='plotly_white', showlegend=False, height=400)
    fig.show()

In [ ]:
# ── Temperature vs AQ scatter with trendline ──
if len(aq_cols) > 0:
    fig, axes = plt.subplots(1, min(3, len(aq_cols)), figsize=(6 * min(3, len(aq_cols)), 5))
    if min(3, len(aq_cols)) == 1: axes = [axes]

    for ax, aq in zip(axes, aq_cols[:3]):
        valid = df[[temp_col, aq]].dropna()
        sample = valid.sample(min(5000, len(valid)), random_state=42)
        ax.scatter(sample[temp_col], sample[aq], alpha=0.1, s=3, color='teal')
        z = np.polyfit(sample[temp_col], sample[aq], 1)
        x_range = np.linspace(sample[temp_col].min(), sample[temp_col].max(), 100)
        ax.plot(x_range, np.poly1d(z)(x_range), 'r-', lw=2, label=f'slope={z[0]:.2f}')
        ax.set_xlabel(temp_col); ax.set_ylabel(aq.replace('air_quality_',''))
        ax.set_title(f'Temp vs {aq.replace("air_quality_","")}', fontsize=11)
        ax.legend(fontsize=9)
    plt.suptitle('Temperature Impact on Air Quality', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

<a id='13'></a>

---
## 13. Spatial & Geographical Analysis

Geographical patterns using proper continent assignment via `pycountry_convert`, Plotly globe visualization, and Folium heatmaps.

---

In [ ]:
# ── Proper continent assignment via pycountry_convert ──
if country_col:
    def get_continent(country_name):
        try:
            alpha2 = pc.country_name_to_country_alpha2(country_name)
            continent_code = pc.country_alpha2_to_continent_code(alpha2)
            return pc.convert_continent_code_to_continent_name(continent_code)
        except (KeyError, Exception):
            return 'Unknown'

    unique_countries = df[country_col].unique()
    country_to_continent = {c: get_continent(c) for c in unique_countries}
    df['continent'] = df[country_col].map(country_to_continent)

    print("Continent Distribution:")
    print(df['continent'].value_counts())
    print(f"\nUnmapped countries: {sum(1 for v in country_to_continent.values() if v == 'Unknown')}")
    unmapped = [k for k, v in country_to_continent.items() if v == 'Unknown']
    if unmapped:
        print(f"Unmapped: {unmapped[:10]}")

In [ ]:
# ── Plotly interactive globe ──
if lat_col and lon_col and location_col:
    loc_avg = df.groupby(location_col).agg({
        lat_col: 'first', lon_col: 'first', temp_col: 'mean',
        country_col: 'first'
    }).reset_index()
    if 'continent' in df.columns:
        cont_map = df.groupby(location_col)['continent'].first()
        loc_avg['continent'] = loc_avg[location_col].map(cont_map)

    fig = px.scatter_geo(loc_avg, lat=lat_col, lon=lon_col, color=temp_col,
                         hover_name=location_col, hover_data=[country_col],
                         color_continuous_scale='RdYlBu_r',
                         projection='natural earth',
                         title='Interactive Globe: Global Temperature Distribution',
                         labels={temp_col: 'Avg Temp (\u00b0C)'})
    fig.update_layout(height=600, template='plotly_white')
    fig.update_geos(showcountries=True, countrycolor='lightgray',
                    showcoastlines=True, coastlinecolor='gray')
    fig.show()

In [ ]:
# ── Folium interactive heatmap ──
if lat_col and lon_col:
    import folium
    from folium.plugins import HeatMap

    heat_data = loc_avg[[lat_col, lon_col, temp_col]].dropna()
    m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')
    HeatMap(heat_data.values.tolist(), radius=10, blur=15, max_zoom=5).add_to(m)
    m.save('temperature_heatmap.html')
    print("Interactive heatmap saved: temperature_heatmap.html")
    m

In [ ]:
# ── Continent comparison (interactive) ──
if 'continent' in df.columns:
    cont_data = df[df['continent'] != 'Unknown']
    cont_order = ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']
    existing = [c for c in cont_order if c in cont_data['continent'].values]

    metrics = []
    for cont in existing:
        subset = cont_data[cont_data['continent'] == cont]
        row = {'Continent': cont, 'Avg Temp': subset[temp_col].mean()}
        if precip_col: row['Avg Precip'] = subset[precip_col].mean()
        if humidity_col: row['Avg Humidity'] = subset[humidity_col].mean()
        metrics.append(row)
    metrics_df = pd.DataFrame(metrics)

    fig = make_subplots(rows=1, cols=3, subplot_titles=['Temperature', 'Precipitation', 'Humidity'])
    fig.add_trace(go.Bar(x=metrics_df['Continent'], y=metrics_df['Avg Temp'],
                         marker_color='coral', name='Temp'), row=1, col=1)
    if 'Avg Precip' in metrics_df.columns:
        fig.add_trace(go.Bar(x=metrics_df['Continent'], y=metrics_df['Avg Precip'],
                             marker_color='skyblue', name='Precip'), row=1, col=2)
    if 'Avg Humidity' in metrics_df.columns:
        fig.add_trace(go.Bar(x=metrics_df['Continent'], y=metrics_df['Avg Humidity'],
                             marker_color='mediumseagreen', name='Humidity'), row=1, col=3)
    fig.update_layout(title='Weather Patterns by Continent', showlegend=False,
                      template='plotly_white', height=450)
    fig.show()

In [ ]:
# ── Country-level statistics ──
if country_col:
    agg_dict_c = {temp_col: ['mean', 'std']}
    if precip_col: agg_dict_c[precip_col] = 'mean'
    if humidity_col: agg_dict_c[humidity_col] = 'mean'

    country_stats = df.groupby(country_col).agg(agg_dict_c).round(2)
    country_stats.columns = ['_'.join(c).strip('_') for c in country_stats.columns]
    country_stats = country_stats.sort_values(country_stats.columns[0], ascending=False)

    print("Top 20 Hottest Countries")
    print("=" * 60)
    print(country_stats.head(20))
    print("\nTop 20 Coldest Countries")
    print("=" * 60)
    print(country_stats.tail(20))

In [ ]:
# ── Latitude-temperature gradient ──
if lat_col:
    lat_bins = pd.cut(df[lat_col], bins=np.arange(-90, 91, 5))
    lat_temp = df.groupby(lat_bins, observed=True)[temp_col].mean().reset_index()
    lat_temp['lat_label'] = lat_temp[lat_col].astype(str)

    fig = px.line(lat_temp, x='lat_label', y=temp_col, markers=True,
                  title='Temperature vs Latitude - Global Gradient',
                  labels={temp_col: 'Avg Temperature (\u00b0C)', 'lat_label': 'Latitude Band'})
    fig.update_layout(template='plotly_white', height=450, xaxis_tickangle=-45)
    fig.show()

<a id='14'></a>

---
## 14. Conclusions & Key Insights

---

In [ ]:
print("""
============================================================================
                       KEY FINDINGS & INSIGHTS
============================================================================

1. DATA CLEANING & PREPROCESSING
   - Missing values handled: median (numeric), mode (categorical)
   - Outliers detected via IQR and Z-score; capped at 1st/99th percentiles
   - Features normalized with StandardScaler

2. EXPLORATORY DATA ANALYSIS
   - Global temperature shows clear seasonal patterns
   - Strong correlations: temperature <-> humidity, wind, UV index
   - Significant variation across countries and climate zones
   - Interactive Plotly visualizations for deeper exploration

3. ANOMALY DETECTION
   - Isolation Forest: ~5% anomalous observations
   - Z-score analysis confirms rare extreme values
   - Anomalies characterized by extreme multi-variable combinations

4. FORECASTING (12 MODELS COMPARED)
   Statistical: ARIMA (auto-selected via pmdarima), SARIMA, Holt-Winters,
                Prophet
   ML: Ridge, Random Forest, XGBoost, Gradient Boosting
   Tuned: XGBoost (Tuned), Random Forest (Tuned)
   Ensemble: Simple Average, Inverse-RMSE Weighted Average
   - ARIMA order auto-selected via AIC minimization (pmdarima)
   - ML models outperform traditional time series on this dataset
   - Hyperparameter tuning (GridSearchCV + TimeSeriesSplit) improved accuracy
   - Weighted ensemble provides the most robust predictions

5. CROSS-VALIDATION
   - 5-fold TimeSeriesSplit validates model stability across time periods
   - Prevents data leakage inherent in random CV for time series

6. RESIDUAL DIAGNOSTICS
   - Residual distribution, Q-Q plot, temporal pattern, and ACF analysis
   - Shapiro-Wilk normality test on ensemble residuals
   - Residual-based 95% prediction intervals (not naive ±RMSE)

7. FEATURE IMPORTANCE
   - Three methods: RF, Permutation, Correlation (consistent train/test split)
   - Latitude, humidity, pressure consistently top predictors

8. CLIMATE ANALYSIS
   - Clear temperature gradient: tropical -> polar
   - Hemispheric patterns inversely correlated (seasons)
   - Tropical regions show lowest temperature variability

9. ENVIRONMENTAL IMPACT
   - Air quality correlates with temperature and wind patterns
   - Higher temperatures tend to worsen pollution indices
   - Wind speed negatively correlates with pollution

10. GEOGRAPHICAL PATTERNS
    - Proper continent mapping via pycountry_convert
    - Interactive globe and heatmap visualizations
    - Continental weather patterns match climatological expectations

11. MODEL PERSISTENCE
    - Best model serialized via joblib for production deployment
    - Feature list saved for reproducible inference pipeline

============================================================================
PM Accelerator Mission: "We are committed to offering free Product
Management education to teenagers from underserved families. Our mission
is to break down financial barriers and achieve educational fairness."
============================================================================
""")

In [ ]:
# ── Final model comparison ──
results_df = pd.DataFrame(results).sort_values('RMSE').drop_duplicates(subset='Model')
print("FINAL MODEL COMPARISON")
print("=" * 75)
print(results_df.to_string(index=False))
print(f"\nBest: {results_df.iloc[0]['Model']} (RMSE: {results_df.iloc[0]['RMSE']})")

In [ ]:
# ── Final interactive comparison ──
fig = px.bar(results_df, x='RMSE', y='Model', orientation='h',
             color='RMSE', color_continuous_scale='RdYlGn_r',
             title='Final Model Ranking by RMSE (lower = better)',
             labels={'RMSE': 'Root Mean Squared Error'})
fig.update_layout(yaxis={'categoryorder': 'total descending'}, template='plotly_white', height=500)
fig.show()

In [ ]:
# ── Save best model for reproducibility ──
import joblib

best_model_name = results_df.iloc[0]['Model']
models_to_save = {}

if 'XGBoost' in best_model_name and 'Tuned' in best_model_name:
    models_to_save['best_model'] = xgb_tuned
elif 'Random Forest' in best_model_name and 'Tuned' in best_model_name:
    models_to_save['best_model'] = rf_tuned
elif 'XGBoost' in best_model_name:
    models_to_save['best_model'] = xgb_model
elif 'Random Forest' in best_model_name:
    models_to_save['best_model'] = rf_ts
elif 'Gradient Boosting' in best_model_name:
    models_to_save['best_model'] = gb_model
elif 'Ridge' in best_model_name:
    models_to_save['best_model'] = ridge

if models_to_save:
    joblib.dump(models_to_save['best_model'], 'best_model.joblib')
    joblib.dump(lag_features, 'model_features.joblib')
    print(f"Saved best model ({best_model_name}) to best_model.joblib")
    print(f"Saved feature list ({len(lag_features)} features) to model_features.joblib")
    print(f"\nTo reload:")
    print(f"  model = joblib.load('best_model.joblib')")
    print(f"  features = joblib.load('model_features.joblib')")